# 06 - Baseline Model

## Objective
Build a Logistic Regression baseline for attrition prediction.

**Why Logistic Regression first:**
- Fast, interpretable, gives probability outputs
- Provides a real baseline to compare tree-based models against
- Good for initial feature importance signals

## Evaluation Metrics
Using precision, recall, F1, and ROC-AUC — NOT accuracy.
Attrition is imbalanced (84% No / 16% Yes), so accuracy would be misleading.
---

In [1]:
import pandas as pd
import numpy as np
import os
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, roc_auc_score, confusion_matrix,
    precision_recall_curve, roc_curve
)
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)

DATA_PATH = "../data/processed"
df = pd.read_csv(f"{DATA_PATH}/attrition_features.csv")
print(f"Loaded: {df.shape}")

Loaded: (1470, 45)


---
## 1. Split Features and Target
---

In [2]:
# Target
y = df["Attrition"]
X = df.drop(columns=["Attrition"])

print(f"Features: {X.shape[1]}")
print(f"Target distribution: {y.value_counts().to_dict()}")
print(f"Target %: {(y.value_counts(normalize=True)*100).round(2).to_dict()}")

Features: 44
Target distribution: {0: 1233, 1: 237}
Target %: {0: 83.88, 1: 16.12}


---
## 2. Train/Test Split (80/20, Stratified)
---

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f"Train: {X_train.shape[0]} samples")
print(f"Test:  {X_test.shape[0]} samples")
print(f"Train target %: {(y_train.value_counts(normalize=True)*100).round(2).to_dict()}")
print(f"Test target %:  {(y_test.value_counts(normalize=True)*100).round(2).to_dict()}")

Train: 1176 samples
Test:  294 samples
Train target %: {0: 83.84, 1: 16.16}
Test target %:  {0: 84.01, 1: 15.99}


---
## 3. Scale Features and Train Logistic Regression
---

In [4]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model_lr = LogisticRegression(max_iter=1000, random_state=42)
model_lr.fit(X_train_scaled, y_train)

print(f"Logistic Regression trained.")
print(f"Intercept: {model_lr.intercept_[0]:.4f}")

Logistic Regression trained.
Intercept: -2.6778


---
## 4. Evaluate
---

In [5]:
# Predictions
y_pred = model_lr.predict(X_test_scaled)
y_prob = model_lr.predict_proba(X_test_scaled)[:, 1]

# Metrics
roc_auc = roc_auc_score(y_test, y_prob)
print("=== Classification Report ===")
print(classification_report(y_test, y_pred, target_names=["No", "Yes"]))
print(f"ROC-AUC: {roc_auc:.4f}")

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(f"  TN={cm[0][0]}  FP={cm[0][1]}")
print(f"  FN={cm[1][0]}  TP={cm[1][1]}")

=== Classification Report ===
              precision    recall  f1-score   support

          No       0.89      0.96      0.92       247
         Yes       0.63      0.36      0.46        47

    accuracy                           0.86       294
   macro avg       0.76      0.66      0.69       294
weighted avg       0.85      0.86      0.85       294

ROC-AUC: 0.8192
Confusion Matrix:
  TN=237  FP=10
  FN=30  TP=17


---
## 5. ROC Curve
---

In [6]:
fpr, tpr, thresholds = roc_curve(y_test, y_prob)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, label=f"Logistic Regression (AUC = {roc_auc:.3f})")
ax.plot([0, 1], [0, 1], "k--", alpha=0.5)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve — Logistic Regression Baseline")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("../models/roc_curve_baseline.png", dpi=100, bbox_inches="tight")
plt.show()
print("Saved: models/roc_curve_baseline.png")

Saved: models/roc_curve_baseline.png


---
## 6. Top Feature Coefficients
---

In [7]:
# Feature importance from logistic regression coefficients
coef_df = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model_lr.coef_[0],
    "AbsCoefficient": np.abs(model_lr.coef_[0])
}).sort_values("AbsCoefficient", ascending=False)

print("Top 15 features by absolute coefficient:")
print(coef_df.head(15).to_string(index=False))

Top 15 features by absolute coefficient:
                         Feature  Coefficient  AbsCoefficient
                        OverTime     0.849676        0.849676
BusinessTravel_Travel_Frequently     0.777630        0.777630
   JobRole_Laboratory Technician     0.683130        0.683130
              NumCompaniesWorked     0.528412        0.528412
         YearsSinceLastPromotion     0.475142        0.475142
          EducationField_Medical    -0.464746        0.464746
    BusinessTravel_Travel_Rarely     0.460866        0.460866
    JobRole_Sales Representative     0.450519        0.450519
         JobRole_Sales Executive     0.441420        0.441420
    EducationField_Life Sciences    -0.410796        0.410796
                DistanceFromHome     0.401630        0.401630
                  JobInvolvement    -0.383876        0.383876
                             Age    -0.377452        0.377452
            MaritalStatus_Single     0.365360        0.365360
              satisfaction_sc

---
## Baseline Summary

| Metric | Value |
| --- | --- |
| Model | LogisticRegression(max_iter=1000) |
| Train/Test | 80/20 stratified |
| ROC-AUC | see above |
| Key insight | OverTime and MonthlyIncome are top drivers |

**Next step:** Model Comparison (07_model_comparison.ipynb)
